In [1]:
!pip install -U ultralytics sahi supervision opencv-python gdown


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.2/207.2 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.8 MB/s eta 0:00:00
   

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files
import datetime
import shutil

uploaded = files.upload()
image_filename = list(uploaded.keys())[0]

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

drive_image_path = f"/content/drive/MyDrive/inference_imgs/input_{timestamp}_{image_filename}"
output_image_name = f"prediction_{timestamp}_{image_filename.split('.')[0]}"
output_image_path = f"/content/drive/MyDrive/inference_imgs/{output_image_name}.png"

shutil.copy(image_filename, drive_image_path)

from ultralytics import YOLOWorld
from sahi.predict import get_prediction, get_sliced_prediction, predict
from sahi import AutoDetectionModel

model_path = "/content/drive/MyDrive/yolov8s_worldv2/runs/weights/last.pt"
model = YOLOWorld(model_path)

detection_model = AutoDetectionModel.from_pretrained(
    model_type='ultralytics',
    model_path="/content/drive/MyDrive/yolov8s_worldv2/runs/weights/last.pt",
    confidence_threshold=0.3,
    device='cuda:0',
)

result = get_sliced_prediction(
    drive_image_path,
    detection_model,
    slice_height = 640,
    slice_width = 640,
    overlap_height_ratio = 0.2,
    overlap_width_ratio = 0.2,

)





In [4]:
from PIL import Image
result.export_visuals(
    export_dir="/content/drive/MyDrive/inference_imgs/",
    text_size=0.5,  # Size of the class label text
    rect_th=1,      # Thickness of bounding box lines
    hide_labels=False,  # Set True to hide class labels
    hide_conf=True,    # Set True to hide confidence scores
    file_name=output_image_name,

)

In [ ]:
import cv2
from google.colab.patches import cv2_imshow
from collections import Counter

img = cv2.imread(output_image_path)
cv2_imshow(img)

class_names = [pred.category.name for pred in result.object_prediction_list]
class_counts = Counter(class_names)

print("\nDetected Object Counts:")
for cls, count in class_counts.items():
    print(f"{cls}: {count}")
